# 07 — Model Training & Evaluation

## AI-Based NIFTY 50 Portfolio Risk Prediction & Early Warning System

**Objective:** train and evaluate models for `Target_DD_5Pct_10D` — at least 5% portfolio drawdown within the next 10 trading days.

**Rules:** chronological splits, no shuffle, preprocessing fit only on training data, validation for model/threshold selection, untouched final test set, no future/target leakage.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix, classification_report, roc_curve, precision_recall_curve

pd.set_option("display.max_columns", 200)
DATA=Path("../data/processed"); REPORTS=Path("../reports"); MODELS=Path("../models")
REPORTS.mkdir(parents=True, exist_ok=True); MODELS.mkdir(parents=True, exist_ok=True)
INPUT_PATH=DATA/"portfolio_ml_features.csv"; TARGET="Target_DD_5Pct_10D"; RANDOM_STATE=42
print(INPUT_PATH)

## 1. Load and inspect the final ML dataset

In [ ]:
df=pd.read_csv(INPUT_PATH,parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
print("Shape:",df.shape)
print("Date range:",df.Date.min(),"->",df.Date.max())
print("Target counts:")
print(df[TARGET].value_counts(dropna=False).sort_index())
print("Target positive rate:",df[TARGET].mean())
df.head()

## 2. Define leakage-safe model features

Notebook 06 established that features are point-in-time and that future/target fields are excluded from the feature set.

In [ ]:
target_cols={"Target_DD_3Pct_10D","Target_DD_5Pct_10D","Target_DD_10Pct_10D"}
leakage_keywords=["future","target","label","forward","next_","lead_"]
model_df=df.dropna(subset=[TARGET]).copy()
model_df[TARGET]=model_df[TARGET].astype(int)
feature_cols=[x for x in model_df.columns if x!="Date" and x not in target_cols and not x.startswith("Future_") and pd.api.types.is_numeric_dtype(model_df[x])]
suspected=[x for x in feature_cols if any(k in x.lower() for k in leakage_keywords)]
assert not suspected,f"Leakage candidates: {suspected}"
assert set(model_df[TARGET].unique()).issubset({0,1})
print("Feature count:",len(feature_cols)); print(feature_cols)

## 3. Chronological 70 / 15 / 15 split

No random shuffling. Train is used for fitting, validation for selection, and test only for final evaluation.

In [ ]:
n=len(model_df); train_end=int(n*.70); valid_end=int(n*.85)
train=model_df.iloc[:train_end].copy(); valid=model_df.iloc[train_end:valid_end].copy(); test=model_df.iloc[valid_end:].copy()
X_train,y_train=train[feature_cols],train[TARGET]
X_valid,y_valid=valid[feature_cols],valid[TARGET]
X_test,y_test=test[feature_cols],test[TARGET]
split_summary=pd.DataFrame({"Split":["Train","Validation","Test"],"Rows":[len(train),len(valid),len(test)],"Start_Date":[train.Date.min(),valid.Date.min(),test.Date.min()],"End_Date":[train.Date.max(),valid.Date.max(),test.Date.max()],"Positive_Events":[y_train.sum(),y_valid.sum(),y_test.sum()],"Positive_Rate":[y_train.mean(),y_valid.mean(),y_test.mean()]})
display(split_summary)
assert train.Date.max()<valid.Date.min()<test.Date.min()
assert all(y.nunique()==2 for y in [y_train,y_valid,y_test])
print("Chronological split checks: PASSED")

## 4. Define candidate models

Imputation/scaling are inside pipelines so validation/test data cannot influence preprocessing. Logistic Regression provides an interpretable baseline; Random Forest and HistGradientBoosting provide nonlinear alternatives.

In [ ]:
models={
"Logistic_Regression":Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler()),("model",LogisticRegression(max_iter=2000,class_weight="balanced",random_state=RANDOM_STATE))]),
"Random_Forest":Pipeline([("imputer",SimpleImputer(strategy="median")),("model",RandomForestClassifier(n_estimators=400,max_depth=6,min_samples_leaf=5,class_weight="balanced",random_state=RANDOM_STATE,n_jobs=-1))]),
"Hist_Gradient_Boosting":Pipeline([("imputer",SimpleImputer(strategy="median")),("model",HistGradientBoostingClassifier(max_iter=300,learning_rate=.05,max_leaf_nodes=15,l2_regularization=1.0,random_state=RANDOM_STATE))])
}
print(list(models))

## 5. Train on Train and compare on Validation

Primary selection metric: **PR-AUC**. ROC-AUC, recall, precision and F1 are retained as diagnostics.

In [ ]:
def metrics(y,p,t=.5):
 q=(p>=t).astype(int)
 return {"ROC_AUC":roc_auc_score(y,p),"PR_AUC":average_precision_score(y,p),"Accuracy":accuracy_score(y,q),"Balanced_Accuracy":balanced_accuracy_score(y,q),"Precision":precision_score(y,q,zero_division=0),"Recall":recall_score(y,q,zero_division=0),"F1":f1_score(y,q,zero_division=0)}

fitted={}; rows=[]
for name,pipe in models.items():
 m=clone(pipe); m.fit(X_train,y_train); fitted[name]=m
 p=m.predict_proba(X_valid)[:,1]; r=metrics(y_valid,p); r.update(Model=name,Split="Validation"); rows.append(r)
validation=pd.DataFrame(rows).set_index("Model").sort_values("PR_AUC",ascending=False)
display(validation)
best_name=validation.index[0]; print("Validation winner:",best_name)

## 6. Select the risk-alert threshold using Validation only

A 0.50 probability threshold is not automatically optimal for an early-warning system. We select the threshold that maximizes validation F1. The test set is still untouched.

In [ ]:
valid_prob=fitted[best_name].predict_proba(X_valid)[:,1]
threshold_rows=[]
for t in np.arange(.10,.91,.05):
 q=(valid_prob>=t).astype(int); threshold_rows.append({"Threshold":round(float(t),2),"Precision":precision_score(y_valid,q,zero_division=0),"Recall":recall_score(y_valid,q,zero_division=0),"F1":f1_score(y_valid,q,zero_division=0),"Balanced_Accuracy":balanced_accuracy_score(y_valid,q)})
threshold_table=pd.DataFrame(threshold_rows)
display(threshold_table)
best_threshold=float(threshold_table.loc[threshold_table.F1.idxmax(),"Threshold"])
print("Selected threshold:",best_threshold)

## 7. Validation ROC and Precision-Recall curves

In [ ]:
fpr,tpr,_=roc_curve(y_valid,valid_prob); pr,rc,_=precision_recall_curve(y_valid,valid_prob)
plt.figure(figsize=(7,5)); plt.plot(fpr,tpr,label=f"ROC-AUC={roc_auc_score(y_valid,valid_prob):.3f}"); plt.plot([0,1],[0,1],"--"); plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate"); plt.title(f"Validation ROC — {best_name}"); plt.legend(); plt.tight_layout(); plt.show()
plt.figure(figsize=(7,5)); plt.plot(rc,pr,label=f"PR-AUC={average_precision_score(y_valid,valid_prob):.3f}"); plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title(f"Validation PR Curve — {best_name}"); plt.legend(); plt.tight_layout(); plt.show()

## 8. Refit selected model on Train + Validation

The model is refit only after model and threshold decisions are fixed. The test period remains untouched until final evaluation.

In [ ]:
train_valid=pd.concat([train,valid]).reset_index(drop=True)
final_model=clone(models[best_name]); final_model.fit(train_valid[feature_cols],train_valid[TARGET])
print("Refit observations:",len(train_valid)); print("Training period:",train_valid.Date.min(),"->",train_valid.Date.max())

## 9. Final untouched Test evaluation

In [ ]:
test_prob=final_model.predict_proba(X_test)[:,1]
test_pred=(test_prob>=best_threshold).astype(int)
test_metrics=metrics(y_test,test_prob,best_threshold)
print("Selected threshold metrics:"); display(pd.DataFrame([test_metrics]))
print("\nConfusion matrix:"); print(confusion_matrix(y_test,test_pred))
print("\nClassification report:"); print(classification_report(y_test,test_pred,target_names=["No 5% DD event","5% DD event"],zero_division=0))

## 10. Final Test ROC and PR curves

In [ ]:
fpr,tpr,_=roc_curve(y_test,test_prob); pr,rc,_=precision_recall_curve(y_test,test_prob)
plt.figure(figsize=(7,5)); plt.plot(fpr,tpr,label=f"ROC-AUC={roc_auc_score(y_test,test_prob):.3f}"); plt.plot([0,1],[0,1],"--"); plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate"); plt.title(f"Test ROC — {best_name}"); plt.legend(); plt.tight_layout(); plt.show()
plt.figure(figsize=(7,5)); plt.plot(rc,pr,label=f"PR-AUC={average_precision_score(y_test,test_prob):.3f}"); plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title(f"Test PR Curve — {best_name}"); plt.legend(); plt.tight_layout(); plt.show()

## 11. Feature importance / coefficients

In [ ]:
est=final_model.named_steps["model"]
if hasattr(est,"feature_importances_"): vals=est.feature_importances_; method="Tree feature importance"
elif hasattr(est,"coef_"): vals=np.abs(est.coef_[0]); method="Absolute logistic coefficient"
else: vals=np.full(len(feature_cols),np.nan); method="Unavailable"
feature_importance=pd.DataFrame({"Feature":feature_cols,"Importance":vals,"Method":method}).sort_values("Importance",ascending=False)
display(feature_importance.head(20))

## 12. Save Notebook 07 outputs

In [ ]:
predictions=pd.DataFrame({"Date":test.Date.values,"Actual_Target_DD_5Pct_10D":y_test.values,"Predicted_Probability":test_prob,"Predicted_Class":test_pred,"Alert_Threshold":best_threshold})
comparison=validation.reset_index(); comparison["Selected_Model"]=comparison.Model.eq(best_name); comparison["Threshold"]=.5
test_row=pd.DataFrame([{"Model":best_name,"Split":"Test","Threshold":best_threshold,**test_metrics}]); comparison=pd.concat([comparison,test_row],ignore_index=True,sort=False)
comparison.to_csv(REPORTS/"model_comparison.csv",index=False)
predictions.to_csv(REPORTS/"test_predictions.csv",index=False)
split_summary.to_csv(REPORTS/"model_split_summary.csv",index=False)
feature_importance.to_csv(REPORTS/"model_feature_importance.csv",index=False)
joblib.dump(fitted,MODELS/"07_model_comparison.joblib")
joblib.dump({"model_name":best_name,"model":final_model,"feature_columns":feature_cols,"target":TARGET,"threshold":best_threshold,"random_state":RANDOM_STATE},MODELS/"07_best_model.joblib")
print("Saved:")
for p in [REPORTS/"model_comparison.csv",REPORTS/"test_predictions.csv",REPORTS/"model_split_summary.csv",REPORTS/"model_feature_importance.csv",MODELS/"07_model_comparison.joblib",MODELS/"07_best_model.joblib"]: print(p)

## 13. Final quality checks

In [ ]:
assert train.Date.max()<valid.Date.min()<test.Date.min()
assert not suspected
assert len(predictions)==len(test)
assert predictions.Predicted_Probability.between(0,1).all()
assert best_name in models and 0<best_threshold<1
print("Selected model:",best_name)
print("Selected threshold:",best_threshold)
print("Validation PR-AUC:",validation.loc[best_name,"PR_AUC"])
print("Test ROC-AUC:",test_metrics["ROC_AUC"])
print("Test PR-AUC:",test_metrics["PR_AUC"])
print("FINAL QUALITY CHECKS: PASSED")

# Final methodology

- Primary target: **5% portfolio drawdown within the next 10 trading days**.
- Chronological 70/15/15 train/validation/test split.
- Validation PR-AUC used for model selection.
- Alert threshold selected using validation data only.
- Test set used only for final out-of-sample evaluation.
- Next notebook: `08_model_comparison_and_selection.ipynb`.